# FAISS RAG Service — Smoke Test

Quick end-to-end verification of the migrated FAISS index:
1. Load the `FaissRagService` from `data/faiss_migrated/`
2. Check index health (vector count, docstore coverage)
3. Run a handful of test queries and inspect returned documents
4. Compare a small batch of queries against Elasticsearch to sanity-check retrieval quality

In [1]:
from __future__ import annotations

import sys
import pathlib

# ── Repo root on sys.path ────────────────────────────────────────────────────
REPO_ROOT = pathlib.Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import os
import logging

from dotenv import load_dotenv
load_dotenv()

logging.basicConfig(level=logging.WARNING)
logging.getLogger("src").setLevel(logging.INFO)

from config import DATA_DIR
from src.rag.faiss_rag_service import FaissRagService
from src.rag.utils import IndexingConfig

FAISS_INDEX_PATH = DATA_DIR / "faiss_migrated"
print(f"Index path : {FAISS_INDEX_PATH}")
print(f"Path exists: {FAISS_INDEX_PATH.exists()}")

Index path : /Users/cyro/Documents/VSC/PopularityBias/data/faiss_migrated
Path exists: True


## 1. Instantiate and load the index

In [ ]:
config = IndexingConfig(
    chunk_size=1000,
    chunk_overlap=100,
    embedding_model="Lajavaness/bilingual-embedding-small",
    embedding_provider="modal",
    request_batch_size=254,
    gpu_batch_size=254,
    normalise_embeddings=True,
    trust_remote_code=True,
)

faiss_service = FaissRagService(
    config=config,
    strategy="ivfpq",
    ivfpq_nprobe=256,
    distance_strategy="cosine",
)

faiss_service.load_index(FAISS_INDEX_PATH)
print("✓ Index loaded")

INFO:src.embeddings.modal_embedding:Initialized ModalEmbeddings with model Lajavaness/bilingual-embedding-small and GPU batch size 254
INFO:src.rag.faiss_rag_service:FAISS ivfpq strategy ready (mmap=True, max_ram=4096 MB)
INFO:src.rag.faiss_rag_service:Loading FAISS index with memory-mapping…
INFO:src.rag.faiss_rag_service:✓ 24,708,051 vectors (mmap)


✓ Index loaded


## 1b. Inspect raw docstore metadata

In [3]:
# Sample 5 docs directly from the docstore (generic — works with any docstore)
import itertools

store = faiss_service._faiss_store
sample_ids = list(itertools.islice(store.index_to_docstore_id.values(), 5))
print(f"Docstore type : {type(store.docstore).__name__}")
print(f"Total docs    : {len(store.docstore):,}\n")
all_keys: set[str] = set()
for uid in sample_ids:
    doc = store.docstore.search(uid)
    all_keys.update(doc.metadata.keys())
    print(f"uid      : {uid}")
    print(f"metadata : {doc.metadata}")
    print(f"content  : {doc.page_content[:80]}…")
    print()
print(f"Metadata keys found: {sorted(all_keys)}")

Sample rows from docstore.sqlite:

uid      : doc_159744
metadata : {'wikipedia_id': 19001859, 'wikipedia_title': 'Thomas Mardy Jones', 'popularity_avg': 42.0625, 'popularity_rank': 3454179.0833333335}
content  : Section::::References.

BULLET::::- The Northern Herald

BULLET::::- The Nationa…

uid      : doc_159745
metadata : {'wikipedia_id': 19001865, 'wikipedia_title': 'Pawelce', 'popularity_avg': 7.808712121212121, 'popularity_rank': 5219102.78125}
content  : Pawelce

Pawelce is a village in the administrative district of Gmina Klonowa, w…

uid      : doc_159746
metadata : {'wikipedia_id': 19001898, 'wikipedia_title': 'Biskupice, Łódź Voivodeship', 'popularity_avg': 6.25, 'popularity_rank': 5312343.916666667}
content  : Biskupice, Łódź Voivodeship

Biskupice is a village in the administrative distri…

uid      : doc_159747
metadata : {'wikipedia_id': 19001868, 'wikipedia_title': 'Świątki, Łódź Voivodeship', 'popularity_avg': 5.856060606060606, 'popularity_rank': 5373601.71875}
cont

## 2. Index health check

In [4]:
store = faiss_service._faiss_store
index = store.index

print(f"Vectors in index : {index.ntotal:,}")
print(f"Docs in docstore : {len(store.docstore):,}")
print(f"Index trained    : {index.is_trained}")
print(f"Strategy         : {faiss_service.strategy}")
print(f"Distance         : {faiss_service.distance_strategy}")
print(f"nprobe           : {faiss_service.ivfpq_nprobe}")

Vectors in index : 24,708,051
Docs in docstore : (sqlite — see below)
Index trained    : True
Strategy         : ivfpq
Distance         : cosine


## 3. Single-query retrieval

In [5]:
# Check id_map coverage (SQLite docstore exposes a cursor)
from src.storage.sqlite_docstore import SqliteDocstore

if isinstance(store.docstore, SqliteDocstore):
    stats = store.docstore.get_stats()
    doc_count   = stats["doc_count"]
    idmap_count = stats["id_map_count"]
    db_mb       = stats["db_size_mb"]
    print(f"SQLite docs      : {doc_count:,}")
    print(f"SQLite id_map    : {idmap_count:,}")
    print(f"DB size          : {db_mb:.1f} MB")
    gap = idmap_count - doc_count
    print(f"Gap (train vecs) : {gap:,}  {'✓ expected' if gap > 0 else '⚠ unexpected'}")
else:
    print("(In-memory docstore — no SQLite stats)")

SQLite docs      : 24,548,307
SQLite id_map    : 24,708,051
DB size          : 24411.7 MB
Gap (train vecs) : 159,744  ✓ expected


## 3. Single-query retrieval

In [6]:
TEST_QUERIES = [
    "Who was the first person to walk on the moon?",
    "What is the capital of France?",
    "When was the Eiffel Tower built?",
    "What programming language was created by Guido van Rossum?",
    "What is the largest country in the world by area?",
]

TOP_K = 3

for query in TEST_QUERIES:
    docs_with_scores = faiss_service.retrieve_documents_with_scores(query, top_k=TOP_K)
    print(f"\nQuery: {query}")
    print("-" * 60)
    for rank, (doc, score) in enumerate(docs_with_scores, 1):
        title = doc.metadata.get("wikipedia_title", "—")
        pop   = doc.metadata.get("popularity", "—")
        snippet = doc.page_content[:120].replace("\n", " ")
        print(f"  [{rank}] score={score:.4f}  title={title!r}  pop={pop}")
        print(f"       {snippet}…")


Query: Who was the first person to walk on the moon?
------------------------------------------------------------
  [1] score=0.4225  title='List of people who have walked on the Moon'  pop=—
       List of people who have walked on the Moon  Twelve people have walked on the Moon, starting with Neil Armstrong and endi…
  [2] score=0.4289  title='NASA'  pop=—
       The first person to stand on the Moon was Neil Armstrong, who was followed 19 minutes later by Buzz Aldrin, while Michae…
  [3] score=0.4329  title='Omega SA'  pop=—
       Section::::Historic events.:First watch on the moon.  The Omega Speedmaster Professional Chronograph was the first watch…

Query: What is the capital of France?
------------------------------------------------------------
  [1] score=0.3909  title='France–Russia relations'  pop=—
       of the Russian Empire was partially the result of a massive influx of French capital into the country.…
  [2] score=0.4016  title='Blois'  pop=—
       Blois  Blois (, ) 

## 4. Batch retrieval timing

In [7]:
import time

BATCH_QUERIES = TEST_QUERIES * 4  # 20 queries

t0 = time.perf_counter()
batch_results = faiss_service.batch_retrieve_with_scores(BATCH_QUERIES, top_k=5)
elapsed = time.perf_counter() - t0

print(f"Queries       : {len(BATCH_QUERIES)}")
print(f"Total time    : {elapsed:.2f}s")
print(f"Per-query avg : {elapsed / len(BATCH_QUERIES) * 1000:.1f}ms")
print(f"Results shape : {len(batch_results)} lists × {len(batch_results[0])} docs each")

Retrieving (ivfpq): 100%|██████████| 20/20 [00:13<00:00,  1.46it/s]

Queries       : 20
Total time    : 13.75s
Per-query avg : 687.3ms
Results shape : 20 lists × 5 docs each


## 5. Side-by-side comparison with Elasticsearch (optional)

Skip this section if ES is not running locally.

In [8]:
ES_URL  = os.getenv("ELASTICSEARCH_ENDPOINT", "")
ES_USER = os.getenv("ELASTICSEARCH_USERNAME", "")
ES_PASS = os.getenv("ELASTICSEARCH_PASSWORD", "")
ES_INDEX = "wiki_full_bil"

RUN_ES_COMPARISON = bool(ES_URL and ES_USER and ES_PASS)
print(f"ES comparison enabled: {RUN_ES_COMPARISON}")
if not RUN_ES_COMPARISON:
    print("  (set ELASTICSEARCH_ENDPOINT / USERNAME / PASSWORD in .env to enable)")

ES comparison enabled: True


In [9]:
if RUN_ES_COMPARISON:
    from src.rag.elasticsearch_rag_service import ElasticsearchRagService

    es_service = ElasticsearchRagService(
        config=config,
        es_url=ES_URL,
        es_user=ES_USER,
        es_password=ES_PASS,
        bm25_b=0,
    )
    es_service.load_index(ES_INDEX)
    print("✓ ES index loaded")

INFO:src.embeddings.modal_embedding:Initialized ModalEmbeddings with model Lajavaness/bilingual-embedding-small and GPU batch size 254
INFO:src.rag.elasticsearch_rag_service:Elasticsearch vector strategy ready


✓ ES index loaded


In [10]:
if RUN_ES_COMPARISON:
    import pandas as pd

    rows = []
    for query in TEST_QUERIES:
        faiss_docs = faiss_service.retrieve_documents_with_scores(query, top_k=5)
        es_docs    = es_service.retrieve_documents_with_scores(query, top_k=5, strategy="approximation")

        faiss_titles = [d.metadata.get("wikipedia_title", "?") for d, _ in faiss_docs]
        es_titles    = [d.metadata.get("wikipedia_title", "?") for d, _ in es_docs]

        overlap = len(set(faiss_titles) & set(es_titles))
        rows.append({
            "query":        query[:50],
            "faiss_top1":   faiss_titles[0] if faiss_titles else "—",
            "es_top1":      es_titles[0]    if es_titles    else "—",
            "overlap@5":    overlap,
        })

    df_cmp = pd.DataFrame(rows)
    print(df_cmp.to_string(index=False))
    print(f"\nMean overlap@5: {df_cmp['overlap@5'].mean():.1f} / 5")

INFO:src.rag.elasticsearch_rag_service:[Embed] Embedding 1 queries (batch_size=512)...
INFO:src.rag.elasticsearch_rag_service:[Embed] ✓ 1 vectors ready
INFO:src.rag.elasticsearch_rag_service:[Embed] Embedding 1 queries (batch_size=512)...
INFO:src.rag.elasticsearch_rag_service:[Embed] ✓ 1 vectors ready
INFO:src.rag.elasticsearch_rag_service:[Embed] Embedding 1 queries (batch_size=512)...
INFO:src.rag.elasticsearch_rag_service:[Embed] ✓ 1 vectors ready
INFO:src.rag.elasticsearch_rag_service:[Embed] Embedding 1 queries (batch_size=512)...
INFO:src.rag.elasticsearch_rag_service:[Embed] ✓ 1 vectors ready
INFO:src.rag.elasticsearch_rag_service:[Embed] Embedding 1 queries (batch_size=512)...
INFO:src.rag.elasticsearch_rag_service:[Embed] ✓ 1 vectors ready


                                             query                                 faiss_top1                                    es_top1  overlap@5
     Who was the first person to walk on the moon? List of people who have walked on the Moon List of people who have walked on the Moon          3
                    What is the capital of France?                    France–Russia relations                  France in the Middle Ages          2
                  When was the Eiffel Tower built?                           Tourism in Paris                               Eiffel Tower          3
What programming language was created by Guido van                           Standard library                           Guido van Rossum          2
 What is the largest country in the world by area?                              Geography Cup                        Periphery countries          2

Mean overlap@5: 2.4 / 5


## 6. Popularity distribution of retrieved docs

Quick sanity check: are retrieved documents spread across popularity deciles or skewed?

In [11]:
import matplotlib.pyplot as plt
import numpy as np

all_popularities = []
for docs_scores in batch_results:
    for doc, _ in docs_scores:
        pop = doc.metadata.get("popularity")
        if pop is not None:
            try:
                all_popularities.append(float(pop))
            except (ValueError, TypeError):
                pass

if all_popularities:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.hist(all_popularities, bins=30, color="#3498db", edgecolor="white")
    ax.set_xlabel("Popularity (log pageviews)")
    ax.set_ylabel("Retrieved doc count")
    ax.set_title("Popularity distribution of FAISS-retrieved documents")
    plt.tight_layout()
    plt.show()
    print(f"Docs with popularity metadata : {len(all_popularities)} / {sum(len(r) for r in batch_results)}")
    print(f"Popularity range              : {min(all_popularities):.2f} – {max(all_popularities):.2f}")
    print(f"Mean popularity               : {np.mean(all_popularities):.2f}")
else:
    print("No popularity metadata found in retrieved docs — check that the index was built with metadata.")

No popularity metadata found in retrieved docs — check that the index was built with metadata.
